# Step 4: Partial Encoder + Projection/Fusion Fine-Tuning

이 노트북은 Step 3의 projection/fusion tuning을 확장해, frozen MiniLM/CLIP feature bank 위에 작은 trainable adapter를 추가합니다. 즉 raw MiniLM/CLIP 전체를 다시 학습하지는 않지만, encoder 출력 공간의 상위 일부를 조정하는 방식으로 partial end-to-end joint fine-tuning에 가까운 구조를 실험합니다.

구조:
- Text: MiniLM feature bank 🔒 → Text partial adapter 🟡 → TextTower 🟡
- Image: CLIP feature bank 🔒 → Image partial adapter 🟡 → ImageTower 🟡
- Tabular: SVD feature bank 🔒 → TabularTower 🟡
- Fusion: CONCAT → Fusion MLP 🟡 → 64D Game Embedding
- Objective: User Embedding · Game Embedding → BPR Loss

출력:
- `game_fusion/emb_game_partial_fusion_tuned_64.npy`
- `game_fusion/emb_game_partial_fusion_tuned_64.csv`


In [8]:
import importlib
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if ROOT.name == "game_fusion":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
LEARNING_RATE = 8e-4
NUM_EPOCHS = 100
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Root path: {ROOT}")
print(f"PyTorch device: {DEVICE}")

Root path: c:\Users\User\26_2_Contest
PyTorch device: cpu


## 1. Feature Bank와 비교 기준 로드

MiniLM/CLIP/SVD 원본 encoder는 frozen feature bank로 유지합니다. Step 4에서는 이 feature bank 위에 residual adapter를 붙여 text/image encoder의 일부를 조정하는 효과를 냅니다.


In [9]:
from tabular_embedding.tabular_tower import load_tabular_bank
import game_fusion.fusion_tower as fusion_tower_module

importlib.reload(fusion_tower_module)
from game_fusion.fusion_tower import (
    PartialFusionBPRModel,
    prepare_bpr_data,
    train_epoch_bpr,
)

text_tower_path = ROOT / "text_data" / "08_text_tower.py"
text_spec = importlib.util.spec_from_file_location("text_tower", text_tower_path)
text_tower_module = importlib.util.module_from_spec(text_spec)
text_spec.loader.exec_module(text_tower_module)
load_text_bank = text_tower_module.load_text_bank

image_tower_path = ROOT / "image_embedding" / "07_image_tower.py"
image_spec = importlib.util.spec_from_file_location("image_tower", image_tower_path)
image_tower_module = importlib.util.module_from_spec(image_spec)
image_spec.loader.exec_module(image_tower_module)
load_image_bank = image_tower_module.load_image_bank

games = pd.read_parquet(ROOT / "Data_process" / "games_metadata_enriched.parquet")
game_ids = games["app_id"].to_numpy()

emb_step1 = np.load(ROOT / "game_fusion" / "emb_game_concat_64.npy").astype(np.float32)
emb_step3 = np.load(ROOT / "game_fusion" / "emb_game_finetuned_64.npy").astype(np.float32)

text_bank, text_id2row = load_text_bank(
    ROOT / "text_data" / "emb_text_minilm",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
image_bank, image_id2row = load_image_bank(
    ROOT / "image_embedding" / "emb_clip_squash",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
tab_bank, tab_id2row = load_tabular_bank(
    ROOT / "tabular_embedding" / "emb_tabular_svd64",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

print("Loaded Step 4 inputs")
print(f"  Games: {len(game_ids):,}")
print(f"  Step 1 frozen fusion: {emb_step1.shape}")
print(f"  Step 3 projection/fusion tuned: {emb_step3.shape}")
print(f"  Text bank: {tuple(text_bank.shape)}")
print(f"  Image bank: {tuple(image_bank.shape)}")
print(f"  Tabular bank: {tuple(tab_bank.shape)}")

[image_tower] Missing images filled with mean vector: 8 items, sample=[np.int64(2381590), np.int64(661700), np.int64(451330), np.int64(1204870), np.int64(1398280)]
Loaded Step 4 inputs
  Games: 50,872
  Step 1 frozen fusion: (50872, 64)
  Step 3 projection/fusion tuned: (50872, 64)
  Text bank: (50872, 384)
  Image bank: (50872, 512)
  Tabular bank: (50872, 64)


## 2. Interaction Data 준비

추천 목적 학습은 BPR loss를 사용합니다. 실제 recommendation interaction 파일이 있으면 그 파일을 사용하고, 없으면 전체 파이프라인 검증을 위해 synthetic data를 생성합니다.


In [10]:
interaction_files = sorted((ROOT / "mvp_recommendation").glob("*recommendations*.csv"))

if interaction_files:
    interactions_df = pd.read_csv(interaction_files[0])
    print(f"Loaded interaction file: {interaction_files[0]}")
else:
    print("Warning: no interaction CSV found. Creating synthetic data for smoke testing.")
    num_synth_users = 100
    rows = []
    sampled_games = np.random.choice(game_ids, size=min(500, len(game_ids)), replace=False)
    for i, app_id in enumerate(sampled_games):
        rows.append({
            "user_id": f"synthetic_user_{i % num_synth_users}",
            "app_id": int(app_id),
            "label": int(np.random.rand() > 0.25),
        })
    interactions_df = pd.DataFrame(rows)

if "label" in interactions_df.columns:
    positive_interactions = interactions_df[interactions_df["label"] > 0].copy()
else:
    positive_interactions = interactions_df.copy()

positive_interactions = positive_interactions[["user_id", "app_id"]].dropna()
positive_interactions["app_id"] = positive_interactions["app_id"].astype(game_ids.dtype, copy=False)

user_ids = sorted(positive_interactions["user_id"].unique())
user_id2idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
game_id2idx = {app_id: idx for idx, app_id in enumerate(game_ids)}

print("Interaction summary")
print(f"  Total rows: {len(interactions_df):,}")
print(f"  Positive rows: {len(positive_interactions):,}")
print(f"  Users: {len(user_id2idx):,}")
print(f"  Games in catalog: {len(game_id2idx):,}")

Interaction summary
  Total rows: 500
  Positive rows: 388
  Users: 100
  Games in catalog: 50,872


## 3. BPR Sample 생성

각 sample은 `(user_idx, positive_game_idx, negative_game_idx)` 형태입니다. negative game은 가능한 한 해당 user가 positive interaction을 남기지 않은 game에서 샘플링합니다.


In [11]:
bpr_data = prepare_bpr_data(
    positive_interactions,
    len(game_ids),
    game_id2idx,
    user_id2idx,
)

if not bpr_data:
    print("Warning: no usable positives. Creating dummy BPR samples.")
    bpr_data = [(0, 0, 1)] if len(game_ids) > 1 else [(0, 0, 0)]

bpr_tensor = torch.tensor(bpr_data, dtype=torch.long)
train_dataset = TensorDataset(
    bpr_tensor[:, 0],
    bpr_tensor[:, 1],
    bpr_tensor[:, 2],
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"BPR samples: {len(bpr_data):,}")
print(f"Batches per epoch: {len(train_loader):,}")

BPR samples: 388
Batches per epoch: 2


## 4. Partial Encoder + Projection/Fusion Model 학습

`PartialFusionBPRModel`은 frozen feature bank 바로 위에 text/image residual adapter를 둡니다. 이 adapter가 도식의 MiniLM/CLIP 🟡 영역에 해당하며, 이후 projection tower와 fusion tower도 함께 학습합니다.


In [12]:
num_users = max(len(user_id2idx), 1)
model = PartialFusionBPRModel(num_users=num_users, embed_dim=64).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
adapter_params = (
    sum(p.numel() for p in model.text_adapter.parameters() if p.requires_grad)
    + sum(p.numel() for p in model.image_adapter.parameters() if p.requires_grad)
)

print("Partial fusion BPR model initialized")
print(f"  Users: {num_users:,}")
print("  Trainable parts: text adapter + image adapter + all projection/fusion towers")
print(f"  Adapter params: {adapter_params:,}")
print(f"  Total trainable params: {trainable_params:,}")

train_losses = []
for epoch in range(NUM_EPOCHS):
    loss = train_epoch_bpr(model, train_loader, optimizer, text_bank, image_bank, tab_bank, DEVICE)
    train_losses.append(loss)
    scheduler.step()

    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | loss={loss:.6f}")

print(f"Training complete. Final loss: {train_losses[-1]:.6f}")

Partial fusion BPR model initialized
  Users: 100
  Trainable parts: text adapter + image adapter + all projection/fusion towers
  Adapter params: 207,712
  Total trainable params: 564,512
Epoch 001/100 | loss=0.692013
Epoch 010/100 | loss=0.400395
Epoch 020/100 | loss=0.282842
Epoch 030/100 | loss=0.239427
Epoch 040/100 | loss=0.225927
Epoch 050/100 | loss=0.219093
Epoch 060/100 | loss=0.216618
Epoch 070/100 | loss=0.215339
Epoch 080/100 | loss=0.214708
Epoch 090/100 | loss=0.213136
Epoch 100/100 | loss=0.214025
Training complete. Final loss: 0.214025


## 5. 64D Game Embedding 생성 및 저장

학습된 adapter/projection/fusion 경로를 모든 game에 적용해 최종 64D game embedding bank를 저장합니다. user embedding은 학습 objective에만 사용하고, export에는 포함하지 않습니다.


In [13]:
model.eval()
all_embeddings = []

with torch.no_grad():
    for start_idx in range(0, len(game_ids), BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, len(game_ids))
        z_text = text_bank[start_idx:end_idx].to(DEVICE)
        z_image = image_bank[start_idx:end_idx].to(DEVICE)
        z_tab = tab_bank[start_idx:end_idx].to(DEVICE)
        game_emb = model.forward_game_only(z_text, z_image, z_tab)
        all_embeddings.append(game_emb.cpu().numpy())

game_embeddings_partial_tuned = np.concatenate(all_embeddings, axis=0).astype(np.float32)

output_dir = ROOT / "game_fusion"
emb_path = output_dir / "emb_game_partial_fusion_tuned_64.npy"
csv_path = output_dir / "emb_game_partial_fusion_tuned_64.csv"

np.save(emb_path, game_embeddings_partial_tuned)
pd.DataFrame({"app_id": game_ids}).to_csv(csv_path, index=False)

norms = np.linalg.norm(game_embeddings_partial_tuned, axis=1)
print("Saved Step 4 embeddings")
print(f"  NPY: {emb_path}")
print(f"  CSV: {csv_path}")
print(f"  Shape: {game_embeddings_partial_tuned.shape}")
print(f"  Dtype: {game_embeddings_partial_tuned.dtype}")
print(f"  Norm mean/std: {norms.mean():.6f} / {norms.std():.6f}")

Saved Step 4 embeddings
  NPY: c:\Users\User\26_2_Contest\game_fusion\emb_game_partial_fusion_tuned_64.npy
  CSV: c:\Users\User\26_2_Contest\game_fusion\emb_game_partial_fusion_tuned_64.csv
  Shape: (50872, 64)
  Dtype: float32
  Norm mean/std: 1.000000 / 0.000000


## 6. Step 1/3/4 Geometry 비교

이 비교는 추천 성능 지표가 아니라 embedding 공간이 정상적으로 생성되었는지 확인하는 sanity check입니다. 실제 성능 평가는 Recall@K, NDCG@K 같은 downstream recommendation metric으로 확인해야 합니다.


In [14]:
sample_size = min(100, len(game_ids))
sample_games = np.random.choice(len(game_ids), sample_size, replace=False)

sim_step1 = cosine_similarity(emb_step1[sample_games]).flatten()
sim_step3 = cosine_similarity(emb_step3[sample_games]).flatten()
sim_step4 = cosine_similarity(game_embeddings_partial_tuned[sample_games]).flatten()

sim_step1 = sim_step1[sim_step1 != 1.0]
sim_step3 = sim_step3[sim_step3 != 1.0]
sim_step4 = sim_step4[sim_step4 != 1.0]

print("Embedding geometry comparison")
print(f"  Step 1 frozen fusion:              mean={sim_step1.mean():.6f}, std={sim_step1.std():.6f}")
print(f"  Step 3 projection/fusion tuned:    mean={sim_step3.mean():.6f}, std={sim_step3.std():.6f}")
print(f"  Step 4 partial encoder+fusion:     mean={sim_step4.mean():.6f}, std={sim_step4.std():.6f}")
print("Step 4 complete")

Embedding geometry comparison
  Step 1 frozen fusion:              mean=0.863756, std=0.049245
  Step 3 projection/fusion tuned:    mean=0.011944, std=0.299131
  Step 4 partial encoder+fusion:     mean=0.009115, std=0.279591
Step 4 complete
